# 06 — Concurrent Stress Test

Goal: send concurrent requests to the vLLM OpenAI-compatible server and measure how throughput scales. This is where vLLM should start showing its value.

In [ ]:
import sys, asyncio, time
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
from vllm_lab.benchmark import benchmark_openai_concurrent
from vllm_lab.utils import load_config

config = load_config(repo_root / 'configs' / 'lab_config.yaml')
base_url = config['vllm_server']['base_url']
model = config['model']['default_name']

In [ ]:
base_prompts = config['benchmark']['prompts']
prompts = [base_prompts[i % len(base_prompts)] for i in range(32)]
concurrency_levels = [1, 2, 4, 8, 16]
all_rows = []

for c in concurrency_levels:
    print('running concurrency', c)
    start = time.perf_counter()
    rows = await benchmark_openai_concurrent(
        base_url=base_url,
        model=model,
        prompts=prompts,
        concurrency=c,
        max_tokens=128,
        temperature=0.0,
    )
    wall_s = time.perf_counter() - start
    total_output_tokens = sum((r.get('output_tokens') or 0) for r in rows)
    for r in rows:
        r['concurrency'] = c
        r['run_wall_s'] = wall_s
        r['aggregate_output_tps'] = total_output_tokens / wall_s if wall_s else 0
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
df.head()

In [ ]:
summary = df.groupby('concurrency').agg(
    requests=('concurrency', 'count'),
    ok=('status_code', lambda s: int((s == 200).sum())),
    latency_mean_s=('latency_s', 'mean'),
    latency_p50_s=('latency_s', 'median'),
    latency_max_s=('latency_s', 'max'),
    aggregate_output_tps=('aggregate_output_tps', 'max'),
).reset_index()
summary

In [ ]:
ax = summary.plot(x='concurrency', y='aggregate_output_tps', marker='o', legend=False)
ax.set_title('Throughput scaling by concurrency')
ax.set_ylabel('aggregate output tokens/s')

In [ ]:
out = repo_root / 'results' / 'vllm_concurrency_stress_notebook.csv'
df.to_csv(out, index=False)
print('wrote', out)

Interpretation: if concurrency increases but throughput does not improve, possible causes include model too small, CPU/client bottleneck, server saturation, max sequence constraints, low max_num_seqs, or insufficient request volume.